# 02 — Text and narrative structure

What this notebook is for: deciding whether the narrative clusters are
real. There are no gold narrative labels, so there is no F1 here and there
will not be one. What replaces it is a combination of four things, none of
which is sufficient alone:

1. **Silhouette** on the clustered points, sampled. Measures separation,
   not correctness.
2. **Noise ratio.** HDBSCAN's willingness to say 'this belongs to nothing'
   is a feature. A very low noise ratio usually means `min_cluster_size` is
   too small and the model is manufacturing narratives out of chatter.
3. **Cluster size distribution.** One cluster holding most of the corpus is
   a failure however good the silhouette looks.
4. **A manual audit of 20 clusters**, rated coherent / mixed / junk by a
   person. This is the only one that measures what we actually care about.

The audit is the deliverable. The other three are triage.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modeling.config import get_settings, run_fingerprint, set_all_seeds
from modeling.io import CorpusReader, ScoredStore

set_all_seeds()
settings = get_settings()
reader = CorpusReader(settings)
store = ScoredStore(settings)

# Every number below is tied to this fingerprint. If a rerun disagrees with
# a committed figure, this block is where the diagnosis starts.
fingerprint = run_fingerprint()
print(f"seed={fingerprint['seed']}  device={fingerprint['device']}  "
      f"corpus={fingerprint['input_manifest_hash']}")

## The corpus going in

Two properties of Phase 1's corpus shape everything downstream, and both
are visible below.

**GDELT records are article metadata, not article text.** Their median
length is around 83 characters. A 30-character headline embeds to something
that clusters on stopwords, so `configs/scoring.yaml` sets a length floor and
those records are excluded from clustering with a logged count rather than
silently degrading the embedding space.

**Reddit is threaded and Mastodon mostly is not.** That asymmetry matters
for the coordination module in notebook 03, not for clustering.

In [ ]:
records = reader.records()
print(f'{len(records):,} records')

summary = records.assign(chars=records['text'].fillna('').str.len()).groupby('source').agg(
    records=('id', 'size'),
    authors=('author_id', 'nunique'),
    median_chars=('chars', 'median'),
    threaded=('parent_id', lambda s: round(s.notna().mean(), 3)),
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
records.assign(chars=records['text'].fillna('').str.len()).boxplot(
    column='chars', by='source', ax=axes[0], grid=False)
axes[0].set_yscale('log'); axes[0].set_title('text length by source'); axes[0].set_xlabel('')
records['source'].value_counts().plot.bar(ax=axes[1])
axes[1].set_title('records per source')
plt.suptitle(''); plt.tight_layout()

## Narratives

Read the coherence column before the size column. A large cluster with low
coherence is a bag of loosely-related posts that HDBSCAN merged because they
were all vaguely political; a small cluster with high coherence is a genuine
shared claim. The UI surfaces coherence for exactly this reason.

`velocity` is the **peak** posts-per-hour, not the mean. A narrative that
produced 200 posts in one hour and then went quiet for a week is the
interesting case, and a lifetime mean erases it completely.

In [ ]:
narratives = store.read('narratives')
if not len(narratives):
    print('No narratives on disk. Run: python -m modeling.cli score cluster')
else:
    display(narratives[['narrative_id', 'label', 'label_source', 'size',
                        'author_count', 'coherence', 'velocity', 'severity']]
            .sort_values('size', ascending=False).head(20))

In [ ]:
if len(narratives):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    narratives['size'].plot.hist(bins=20, ax=axes[0]); axes[0].set_title('cluster size')
    narratives['coherence'].plot.hist(bins=20, ax=axes[1]); axes[1].set_title('coherence')
    axes[2].scatter(narratives['size'], narratives['coherence'], alpha=0.6)
    axes[2].set_xlabel('size'); axes[2].set_ylabel('coherence')
    axes[2].set_title('big clusters are usually less coherent')
    plt.tight_layout()

## The manual audit

Run the cell, read the twenty clusters, and rate each one:

- **coherent** — the members share one claim
- **mixed** — two or three distinct claims got merged
- **junk** — no shared claim; the cluster is an artefact

Write the counts into `artifacts/error_analysis/cluster.md`. A junk rate
above roughly a fifth means `min_cluster_size` is too permissive for this
corpus.

In [ ]:
from modeling.text.cluster import audit_table

print(audit_table(20))

## Labels: LLM or centroid?

`label_source` says which. With no `ANTHROPIC_API_KEY` the pipeline falls
back to the representative post's first sentence — a worse label and an
honest one. A centroid label reads visibly as a quotation rather than as a
generated headline, which is the right signal to a reader that no model
wrote it.

In [ ]:
if len(narratives):
    print(narratives['label_source'].value_counts(dropna=False))
    for row in narratives.head(5).to_dict('records'):
        print(f"\n[{row['label_source']}] {row['label']}")
        if row.get('summary'):
            print(f"    {row['summary']}")